# Etapa 2 — Descoberta de Padrões e Anomalias com Matrix Profile (`stumpy`)

**Objetivo:** Analisar a série representativa **`MT_206`** (medóide do cluster majoritário da Etapa 1) via Matrix Profile (`stumpy.stump`), comparando janelas ajustadas à periodicidade física ($m=168\text{ h}$, $m=24\text{ h}$ e $m=7\text{ dias}$), extraindo e interpretando *motifs* e *discords*, e demonstrando o papel da normalização Z-score contra falsos positivos de escala e *offset*.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "Codigo" else Path.cwd()
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from Codigo.src.matrix_profile import (
    CLIENTE_REPRESENTATIVO,
    JANELA_DIARIA_SEMANAL_M,
    JANELA_HORARIA_DIARIA_M,
    JANELA_HORARIA_SEMANAL_M,
    SEMENTE_PADRAO,
    TOP_K_PADROES,
    calcular_ou_carregar_mp,
    executar_experimento_zscore,
    extrair_top_discords,
    extrair_top_motifs,
    gerar_figuras_matrix_profile,
    salvar_saidas_matrix_profile,
)
from Codigo.src.preparacao import carregar_serie_tratada

np.random.seed(SEMENTE_PADRAO)
print(f"Raiz do projeto: {RAIZ_PROJETO} | Série representativa: {CLIENTE_REPRESENTATIVO}")

## 1. Carga da Série Representativa e Cálculo do Matrix Profile Multi-Escala

In [ ]:
df_horario = carregar_serie_tratada(
    RAIZ_PROJETO / "Dados/tratado/consumo_horario_clientes_completos_2011_2014.parquet", freq="h"
)
df_diario = carregar_serie_tratada(
    RAIZ_PROJETO / "Dados/tratado/consumo_diario_clientes_completos_2011_2014.parquet", freq="D"
)

serie_h = df_horario[CLIENTE_REPRESENTATIVO].copy()
serie_d = df_diario[CLIENTE_REPRESENTATIVO].copy()

resumo_serie = {
    "cliente": CLIENTE_REPRESENTATIVO,
    "n_horas": int(len(serie_h)),
    "n_dias": int(len(serie_d)),
    "inicio": str(serie_h.index.min()),
    "fim": str(serie_h.index.max()),
    "nans_horario": int(serie_h.isna().sum()),
    "nans_diario": int(serie_d.isna().sum()),
    "media_kw": round(float(serie_h.mean()), 2),
    "desvio_kw": round(float(serie_h.std()), 2),
    "min_kw": round(float(serie_h.min()), 2),
    "mediana_kw": round(float(serie_h.median()), 2),
    "max_kw": round(float(serie_h.max()), 2),
}

mp168_dist, mp168_idx, est_168 = calcular_ou_carregar_mp(
    serie_h, m=JANELA_HORARIA_SEMANAL_M, normalizar=True,
    caminho_cache=RAIZ_PROJETO / "Resultados/mp_horario_m168_zscore.npy",
)
mp24_dist, mp24_idx, est_24 = calcular_ou_carregar_mp(
    serie_h, m=JANELA_HORARIA_DIARIA_M, normalizar=True,
    caminho_cache=RAIZ_PROJETO / "Resultados/mp_horario_m24_zscore.npy",
)
mp7d_dist, mp7d_idx, est_7d = calcular_ou_carregar_mp(
    serie_d, m=JANELA_DIARIA_SEMANAL_M, normalizar=True,
    caminho_cache=RAIZ_PROJETO / "Resultados/mp_diario_m7_zscore.npy",
)

estatisticas_mp = {"horario_m168": est_168, "horario_m24": est_24, "diario_m7": est_7d}
display(pd.DataFrame(estatisticas_mp).T)

## 2. Extração dos Motifs e Discords

In [ ]:
motifs_por_janela = {
    "horario_m168": extrair_top_motifs(serie_h, mp168_dist, mp168_idx, JANELA_HORARIA_SEMANAL_M, TOP_K_PADROES),
    "horario_m24": extrair_top_motifs(serie_h, mp24_dist, mp24_idx, JANELA_HORARIA_DIARIA_M, TOP_K_PADROES),
    "diario_m7": extrair_top_motifs(serie_d, mp7d_dist, mp7d_idx, JANELA_DIARIA_SEMANAL_M, TOP_K_PADROES),
}
discords_por_janela = {
    "horario_m168": extrair_top_discords(serie_h, mp168_dist, mp168_idx, JANELA_HORARIA_SEMANAL_M, TOP_K_PADROES),
    "horario_m24": extrair_top_discords(serie_h, mp24_dist, mp24_idx, JANELA_HORARIA_DIARIA_M, TOP_K_PADROES),
    "diario_m7": extrair_top_discords(serie_d, mp7d_dist, mp7d_idx, JANELA_DIARIA_SEMANAL_M, TOP_K_PADROES),
}

print("--- Top Motifs (m=168h) ---")
display(pd.DataFrame(motifs_por_janela["horario_m168"]))
print("--- Top Discords (m=168h) ---")
display(pd.DataFrame(discords_por_janela["horario_m168"]))

## 3. Papel da Normalização Z-score e Geração de Figuras

In [ ]:
exp_zscore = executar_experimento_zscore(
    serie_horaria=serie_h,
    m=JANELA_HORARIA_SEMANAL_M,
    idx_alvo=motifs_por_janela["horario_m168"][0]["idx_a"],
    mp_norm_precomp=(mp168_dist, mp168_idx),
    caminho_cache_sem_norm=RAIZ_PROJETO / "Resultados/mp_horario_m168_sem_norm.npy",
)
print(json.dumps(exp_zscore["experimento_perturbacao"]["delta_abs_zscore"], indent=2))

figuras = gerar_figuras_matrix_profile(
    serie_horaria=serie_h,
    serie_diaria=serie_d,
    mp_168=(mp168_dist, mp168_idx),
    mp_24=(mp24_dist, mp24_idx),
    mp_7d=(mp7d_dist, mp7d_idx),
    motifs_168=motifs_por_janela["horario_m168"],
    motifs_24=motifs_por_janela["horario_m24"],
    motifs_7d=motifs_por_janela["diario_m7"],
    discords_168=discords_por_janela["horario_m168"],
    discords_24=discords_por_janela["horario_m24"],
    exp_zscore=exp_zscore,
    diretorio_figuras=RAIZ_PROJETO / "Documentacao/figuras",
)
figuras_rel = [str(Path(f).relative_to(RAIZ_PROJETO)) for f in figuras]
salvar_saidas_matrix_profile(
    CLIENTE_REPRESENTATIVO, resumo_serie, estatisticas_mp,
    motifs_por_janela, discords_por_janela, exp_zscore,
    figuras_rel, RAIZ_PROJETO / "Resultados",
)

for fig_path in figuras:
    display(Image(filename=fig_path))